# Notebook 1 - Download 13F Filings from SEC

so this is where everything starts. the SEC has a free API that gives you all the filing data for any fund manager. you just need their CIK number which is basically their unique ID on EDGAR.

i started with berkshire hathaway because i am a big fan of warren buffett but the code works for literally any fund manager, just change the CIK in config.py

what this notebook does:
- hits the SEC submissions API with the manager CIK
- finds all 13F filings across the last N quarters  
- for each filing, figures out which XML file is the actual holdings table (this was trickier than i expected)
- downloads and caches the XML so we dont have to hit the API again
- saves everything to filings.parquet

one thing i learned the hard way - the SEC filing directory has multiple XML files and only one of them has the actual holdings. the other one is just a cover page. so we have to check the content of each file to figure out which one we actually want

In [ ]:
import os, sys, pathlib

# ----- COLAB SETUP -----
# Option A: with Google Drive (data stays between sessions)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: without Drive (data gone when session ends, need to rerun)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# detects root automatically when running locally
PROJECT_ROOT = pathlib.Path("/content/13F-Analytics") if pathlib.Path("/content/13F-Analytics").exists() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("working directory:", PROJECT_ROOT)

In [ ]:
# install if running on colab
# !pip install pyarrow requests pandas matplotlib numpy -q

# set your name and email - SEC rejects requests without this
# !sed -i 's/Your Name your_email@example.com/Your Actual Name youremail@email.com/' src/config.py

In [ ]:
from src import config
from src.utils import make_session
from src.sec import build_filings_dataset

print("manager:", config.MANAGER_NAME)
print("CIK:", config.CIK)
print("quarters to fetch:", config.N_QUARTERS)
print("user agent:", config.USER_AGENT)

In [ ]:
# this will take a few minutes, its downloading 8 quarters of XML from SEC
session = make_session()
filings = build_filings_dataset(session, n_quarters=config.N_QUARTERS)
filings

In [ ]:
# make sure every quarter has a verified XML file
# if any show None here something went wrong with that filing
filings[["quarter", "form", "filingDate", "reportDate", "xml_file"]]

outputs saved to data/processed/filings.parquet and XML cached in data/raw_xml/ - next run notebook 2